# Build chessbench-full (multi-session, resumable)

Processes ChessBench train shards one at a time. Progress (per-shard npz + 9M teacher labels) is uploaded to HF after EVERY shard, so a killed kernel resumes from the manifest and never loses more than one shard. Run this notebook repeatedly until all shards are done.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/chess-slm-benchmark')
SL_REPO = Path('/kaggle/working/searchless_chess')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
except Exception as exc:
    print('HF secret unavailable:', exc)
assert os.environ.get('HF_WRITE_TOKEN'), 'create a Kaggle secret named HF_WRITE_TOKEN'
os.chdir(REPO)
print('repo:', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
# LOCALLY-VALIDATED era stack (jax 0.4.35 + orbax 0.5.5 read the 2024 ocdbt checkpoints).
# jax 0.4.35 has no CUDA wheels on PyPI; GPU comes from the cuda12 pjrt plugin on GCS.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '--quiet',
                'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'chex', 'dm-haiku'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-f', 'https://storage.googleapis.com/jax-releases/jax_releases.html',
                'jax==0.4.35', 'jaxlib==0.4.35', 'jax_cuda12_pjrt==0.4.35', 'jax_cuda12_plugin==0.4.35',
                'orbax-checkpoint==0.5.5', 'dm-haiku==0.0.11', 'numpy==1.26.4',
                'jaxtyping', 'typing-extensions', 'python-chess', 'zstandard',
                'huggingface_hub'], check=True)
print('era stack installed')

In [ ]:
# 9M teacher checkpoint (dim 256 / layers 8 / heads 8)
CK = Path('/kaggle/working/checkpoints')
CK.mkdir(exist_ok=True)
TEACHER = CK / '9M/6400000/params_ema'
if not TEACHER.exists():
    z = CK / '9M.zip'
    subprocess.run(['curl', '-sL', '--retry', '5', '-o', str(z),
                    'https://storage.googleapis.com/searchless_chess/checkpoints/9M.zip'], check=True)
    subprocess.run(['unzip', '-o', '-q', str(z), '-d', str(CK)], check=True)
    z.unlink()
assert TEACHER.exists(), TEACHER
print('9M teacher ready')

In [ ]:
# Production: process the next unfinished shard(s). Resume-safe; rerun this cell
# (or restart the kernel) to continue. N_SHARDS = total target dataset size.
N_SHARDS = 8
cmd = [sys.executable, 'scripts/build_full_dataset.py',
       '--n-shards', str(N_SHARDS),
       '--sl-repo', str(SL_REPO),
       '--workdir', '/kaggle/working/chessbench-build',
       '--teacher-checkpoint', str(TEACHER),
       '--teacher-dim', '256', '--teacher-layers', '8', '--teacher-heads', '8',
       '--hf-repo', 'vedangfake/chess-slm-benchmark',
       '--hf-run', 'chessbench-full-build',
       '--resume-from-hf']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('ALL SHARDS DONE')

After all shards are on HF, assemble the final Kaggle dataset with `scripts/assemble_full_dataset.py` (see runbook).